## Simple implemenation of Bifrost


**Installing Env Variables**

In [2]:
import os
from dotenv import load_dotenv

# Load secrets from .env — never hardcode API keys in this notebook.
# Re-run this cell after editing .env (no kernel restart needed).
load_dotenv(override=True)

REQUIRED_ENV_VARS = (
    "OPENAI_API_KEY",
    "GROQ_API_KEY",
    "BIFROST_OPENAI_API_KEY",
    "BIFROST_GROQ_API_KEY",
    "BIFROST_BASE_URL"
)

missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Copy .env.example to .env and set your values."
    )

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BIFROST_OPENAI_API_KEY = os.getenv("BIFROST_OPENAI_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_GROQ_API_KEY = os.getenv("BIFROST_GROQ_API_KEY")
BIFROST_BASE_URL = os.getenv("BIFROST_BASE_URL")

print("Environment loaded.")


Environment loaded.


**Health Check**

In [3]:
import httpx
try:
    health = httpx.get(f"{BIFROST_BASE_URL}/health")
    print(health.json())
except Exception as e:
    print(f"Error: {e}")


{'components': {'db_pings': 'ok'}, 'status': 'ok'}


### Reusable Text Prompts and Utilits

In [19]:
### Reusable Text Prompts and Utilits
TEST_PROMPTS = {
    "simple": "What is the capital of France?",
    "reasoning": "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code": "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki": "What are the stream modes in the new langgraph version? Use the deepwiki",
    "tavily": "Search the web for the latest news about Groq AI and summarize the top"
}

import time
def time_caputer(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        return result, end_time - start_time
    return wrapper  # <-- this was missing


CHATOPENAI_MODEL = "openai/gpt-4o-mini"
CHATOPENAI_FALLBACK_MODEL = "openai/gpt-5.6-luna"

GROQ_MODEL = "openai/gpt-oss-20b"
GROQ_FALLBACK_MODEL = "openai/gpt-oss-120b"


**Bad Approach Calling direct model**

No caching , routing , virtual keys can be applied , for this we need to make extra efforts 

In [25]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    streaming=True
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses metrics, logs, and traces, enabling developers and operators to gain insights into the performance and behavior of their systems. As cloud-native architectures and microservices become increasingly prevalent, the need for effective observability tools has grown, making OpenTelemetry a vital component in modern software development.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, allowing organizations to instrument their applications without being locked into a specific monitoring solution. This flexibility enables teams to choose the best tools for their needs while maintaining a consistent approach to data collection. OpenTelemetry supports multiple programming languages, making it accessible for diverse technology stacks.

The framework promotes best practices in observability, encou

## Using the BiFrost LLM Gateway

In [32]:
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(
    model=CHATOPENAI_MODEL,              # "openai/gpt-4o-mini"
    base_url=f"{BIFROST_BASE_URL}/langchain",
    api_key=BIFROST_OPENAI_API_KEY,      # not GROQ key
    temperature=0,
    streaming=True,
)

@time_caputer
def getResponse(prompt):
    response_content = ""
    for chunk in llm.stream(prompt):
        text = chunk.content or ""
        response_content += text
        print(text, end="", flush=True)
    return response_content

response, elapsed = getResponse("Give 200 words essay on OpenTelemetry")
print(f"\nElapsed: {elapsed:.2f}s")

OpenTelemetry is an open-source observability framework designed to provide a standardized way to collect, process, and export telemetry data from applications. It encompasses three primary types of telemetry: traces, metrics, and logs, enabling developers to gain comprehensive insights into their systems' performance and behavior. By offering a unified approach, OpenTelemetry simplifies the integration of observability into diverse programming languages and platforms, fostering consistency across various environments.

One of the key advantages of OpenTelemetry is its vendor-agnostic nature, allowing organizations to choose their preferred backend for data analysis without being locked into a specific vendor's ecosystem. This flexibility encourages innovation and enables teams to adapt their observability strategies as their needs evolve. Additionally, OpenTelemetry supports a wide range of instrumentation libraries, making it easier for developers to instrument their applications wit